In [1]:
from pathlib import Path
from urllib.parse import unquote, urlparse
import hashlib
import re
import time

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OPM_EXPORT_XLSX = PROJECT_ROOT / "data" / "sample_sources" / "Collective Bargaining Agreements 2026-04-26_083048.xlsx"

SOURCE_CSV = PROJECT_ROOT / "data" / "sample_sources" / "opm_cba_sources.csv"
MANIFEST_CSV = PROJECT_ROOT / "data" / "sample_sources" / "opm_download_manifest.csv"

RAW_PDF_DIR = PROJECT_ROOT / "data" / "samples" / "raw_pdfs"
RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(OPM_EXPORT_XLSX.exists(), OPM_EXPORT_XLSX)




/Users/mikifoster/Documents/Code_Projects/cba-clock
True /Users/mikifoster/Documents/Code_Projects/cba-clock/data/sample_sources/Collective Bargaining Agreements 2026-04-26_083048.xlsx


In [2]:
opm_export = pd.read_excel(OPM_EXPORT_XLSX)

opm_export.columns = [
    re.sub(r"\s+", "_", str(col).strip().lower().replace("/", "_"))
    for col in opm_export.columns
]

opm_export.head()

,agency,sub-agency___component,activity___office___region,union,local,bus_code(s),expiration_date
0,Chemical Safety/Hazard Investigation Bd,NaN,NaN,AFGE- American Federation of Government Employees,2211,5988,3/20/2029
1,Commodity Futures Trading Commission,NaN,"Washington, DC; Chicago, IL; Kansas City, MO; ...",NTEU- National Treasury Employees Union,337,5938,6/10/2022
2,Consumer Financial Protection Bureau,NaN,NaN,NTEU- National Treasury Employees Union,335,5899,11/08/2028
3,Consumer Product Safety Commission,NaN,NaN,AFGE- American Federation of Government Employees,3579,1062,5/16/2025
4,Court Services and Offender Supervision Agency,Pretrial Services Agency for the District of C...,NaN,AFGE- American Federation of Government Employees,1456,2432,4/05/2020


In [3]:
SEARCH_QUERIES = [
    "site:opm.gov/cba/api/documents attachments pdf collective bargaining agreement",
    "site:opm.gov/cba/api/documents attachments pdf AFGE CBA",
    "site:opm.gov/cba/api/documents attachments pdf NAGE CBA",
    "site:opm.gov/cba/api/documents attachments pdf Treasury AFGE",
    "site:opm.gov/cba/api/documents attachments pdf Army AFGE",
]

In [4]:
scraped_urls = [
    "https://www.opm.gov/cba/api/documents/4ba838b6-8141-496d-981a-62382e7850fb/attachments/ARS%20Animal%20Disease%20Center2015-06-29BUS%201529.pdf",
    "https://www.opm.gov/cba/api/documents/c4ee5660-1499-e911-915b-005056a577c8/attachments/1085_FMCS%20%26%20NAGE%20R3-118_11102020-redacted.pdf",
    "https://www.opm.gov/cba/api/documents/79fa136d-7297-e911-915b-005056a577c8/attachments/2679_DOT%20SLSDC%20%26%20AFGE%201968_09302021-redacted.pdf",
    "https://www.opm.gov/cba/api/documents/6cf9956b-ec93-4bf1-8137-6fa3dd4a355c/attachments/304536845863ArmyUSARAKIMCOMCBA06262006.pdf",
    "https://www.opm.gov/cba/api/documents/5f1837cd-caa7-e911-915c-005056a577c8/attachments/1555%20%26%201564_DOE%20AFGE%20788_06302019-%20redacted.pdf",
    "https://www.opm.gov/cba/api/documents/9c8c526e-f9fb-e911-9160-005056a577c8/attachments/3708_Army_USACE_CBA_02022020.pdf",
    "https://www.opm.gov/cba/api/documents/0d3c0948-d1a4-e911-915b-005056a577c8/attachments/2376_Treasury%20Mint%20FOP_CBA%20updated_Redacted.pdf",
    "https://www.opm.gov/cba/api/documents/c4aed959-fe98-e911-915b-005056a577c8/attachments/2672_DOI%20NPS%20Independence%20NHP%20%26%20AFGE%202058_08122019.pdf",
    "https://www.opm.gov/cba/api/documents/622e1554-2e9c-e911-915b-005056a577c8/attachments/1012_DOI%20BIA%20%26%20Indian%20Educators%20Federation%204524_10042019-%20redacted.pdf",
    "https://www.opm.gov/cba/api/documents/7512a833-af8b-e911-9158-005056a577c8/attachments/1501_Treasury%20BEP%20%26%20IPPDE%20Local%2032%20Engravers_05142022-redacted.pdf",
    "https://www.opm.gov/cba/api/documents/6acdd03c-1e9c-e911-915b-005056a577c8/attachments/1010%201120%20HUD%20%26%20AFGE-%20redacted.pdf",
]

In [5]:
def parse_opm_pdf_url(url: str) -> dict:
    filename = unquote(Path(urlparse(url).path).name)

    document_id_match = re.search(r"/documents/([^/]+)/attachments/", url)
    document_id = document_id_match.group(1) if document_id_match else None

    bus_codes = re.findall(r"(?:BUS\s*|^|\D)(\d{4})(?:\D|$)", filename, flags=re.IGNORECASE)
    date_matches = re.findall(r"\d{4}-\d{2}-\d{2}|\d{8}", filename)

    parsed_date = None
    if date_matches:
        raw_date = date_matches[-1]
        if "-" in raw_date:
            parsed_date = raw_date
        elif len(raw_date) == 8:
            parsed_date = f"{raw_date[4:8]}-{raw_date[0:2]}-{raw_date[2:4]}"

    return {
        "pdf_url": url,
        "document_id": document_id,
        "filename": filename,
        "parsed_bus_codes": ";".join(sorted(set(bus_codes))),
        "parsed_date": parsed_date,
    }


url_df = pd.DataFrame([parse_opm_pdf_url(url) for url in scraped_urls])
url_df

,pdf_url,document_id,filename,parsed_bus_codes,parsed_date
0,https://www.opm.gov/cba/api/documents/4ba838b6...,4ba838b6-8141-496d-981a-62382e7850fb,ARS Animal Disease Center2015-06-29BUS 1529.pdf,1529;2015,2015-06-29
1,https://www.opm.gov/cba/api/documents/c4ee5660...,c4ee5660-1499-e911-915b-005056a577c8,1085_FMCS & NAGE R3-118_11102020-redacted.pdf,1085,2020-11-10
2,https://www.opm.gov/cba/api/documents/79fa136d...,79fa136d-7297-e911-915b-005056a577c8,2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf,1968;2679,2021-09-30
3,https://www.opm.gov/cba/api/documents/6cf9956b...,6cf9956b-ec93-4bf1-8137-6fa3dd4a355c,304536845863ArmyUSARAKIMCOMCBA06262006.pdf,,2006-06-26
4,https://www.opm.gov/cba/api/documents/5f1837cd...,5f1837cd-caa7-e911-915c-005056a577c8,1555 & 1564_DOE AFGE 788_06302019- redacted.pdf,1555;1564,2019-06-30
5,https://www.opm.gov/cba/api/documents/9c8c526e...,9c8c526e-f9fb-e911-9160-005056a577c8,3708_Army_USACE_CBA_02022020.pdf,3708,2020-02-02
6,https://www.opm.gov/cba/api/documents/0d3c0948...,0d3c0948-d1a4-e911-915b-005056a577c8,2376_Treasury Mint FOP_CBA updated_Redacted.pdf,2376,NaN
7,https://www.opm.gov/cba/api/documents/c4aed959...,c4aed959-fe98-e911-915b-005056a577c8,2672_DOI NPS Independence NHP & AFGE 2058_0812...,2058;2672,2019-08-12
8,https://www.opm.gov/cba/api/documents/622e1554...,622e1554-2e9c-e911-915b-005056a577c8,1012_DOI BIA & Indian Educators Federation 452...,1012;4524,2019-10-04
9,https://www.opm.gov/cba/api/documents/7512a833...,7512a833-af8b-e911-9158-005056a577c8,1501_Treasury BEP & IPPDE Local 32 Engravers_0...,1501,2022-05-14


In [6]:
def find_column(df: pd.DataFrame, contains: str) -> str:
    matches = [col for col in df.columns if contains in col]
    if not matches:
        raise ValueError(f"No column found containing: {contains}. Available: {df.columns.tolist()}")
    return matches[0]


bus_col = find_column(opm_export, "bus")
agency_col = find_column(opm_export, "agency")
union_col = find_column(opm_export, "union")
local_col = find_column(opm_export, "local")
expiration_col = find_column(opm_export, "expiration")

print(bus_col, agency_col, union_col, local_col, expiration_col)

bus_code(s) agency union local expiration_date


In [7]:
def normalize_bus_codes(value) -> list[str]:
    if pd.isna(value):
        return []
    return re.findall(r"\d{4}", str(value))


opm_export_work = opm_export.copy()
opm_export_work["normalized_bus_code"] = opm_export_work[bus_col].apply(normalize_bus_codes)

opm_exploded = opm_export_work.explode("normalized_bus_code")
opm_exploded = opm_exploded[opm_exploded["normalized_bus_code"].notna()]
opm_exploded.head()

,agency,sub-agency___component,activity___office___region,union,local,bus_code(s),expiration_date,normalized_bus_code
0,Chemical Safety/Hazard Investigation Bd,NaN,NaN,AFGE- American Federation of Government Employees,2211,5988,3/20/2029,5988
1,Commodity Futures Trading Commission,NaN,"Washington, DC; Chicago, IL; Kansas City, MO; ...",NTEU- National Treasury Employees Union,337,5938,6/10/2022,5938
2,Consumer Financial Protection Bureau,NaN,NaN,NTEU- National Treasury Employees Union,335,5899,11/08/2028,5899
3,Consumer Product Safety Commission,NaN,NaN,AFGE- American Federation of Government Employees,3579,1062,5/16/2025,1062
4,Court Services and Offender Supervision Agency,Pretrial Services Agency for the District of C...,NaN,AFGE- American Federation of Government Employees,1456,2432,4/05/2020,2432


In [8]:
url_work = url_df.copy()
url_work["normalized_bus_code"] = url_work["parsed_bus_codes"].str.split(";")
url_exploded = url_work.explode("normalized_bus_code")
url_exploded = url_exploded[url_exploded["normalized_bus_code"].notna()]
url_exploded = url_exploded[url_exploded["normalized_bus_code"] != ""]

joined = url_exploded.merge(
    opm_exploded,
    on="normalized_bus_code",
    how="left",
    suffixes=("_url", "_opm"),
)

joined[
    [
        "filename",
        "normalized_bus_code",
        agency_col,
        union_col,
        local_col,
        expiration_col,
        "pdf_url",
    ]
].head(20)

,filename,normalized_bus_code,agency,union,local,expiration_date,pdf_url
0,ARS Animal Disease Center2015-06-29BUS 1529.pdf,1529,Department of Agriculture,AFGE- American Federation of Government Employees,2315,6/29/2025,https://www.opm.gov/cba/api/documents/4ba838b6...
1,ARS Animal Disease Center2015-06-29BUS 1529.pdf,2015,NaN,NaN,NaN,NaN,https://www.opm.gov/cba/api/documents/4ba838b6...
2,1085_FMCS & NAGE R3-118_11102020-redacted.pdf,1085,Federal Mediation And Conciliation Service,NAGE- National Association of Government Emplo...,R3-118,2/11/2025,https://www.opm.gov/cba/api/documents/c4ee5660...
3,2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf,1968,Department of Defense,AFGE- American Federation of Government Employees,1858,10/17/2020,https://www.opm.gov/cba/api/documents/79fa136d...
4,2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf,2679,Department of Transportation,AFGE- American Federation of Government Employees,1968,9/30/2028,https://www.opm.gov/cba/api/documents/79fa136d...
5,1555 & 1564_DOE AFGE 788_06302019- redacted.pdf,1555,Department of Energy,AFGE- American Federation of Government Employees,788,9/28/2023,https://www.opm.gov/cba/api/documents/5f1837cd...
6,1555 & 1564_DOE AFGE 788_06302019- redacted.pdf,1564,Department of Energy,AFGE- American Federation of Government Employees,788,9/28/2023,https://www.opm.gov/cba/api/documents/5f1837cd...
7,3708_Army_USACE_CBA_02022020.pdf,3708,Department of Defense,AFGE- American Federation of Government Employees,2187,2/02/2020,https://www.opm.gov/cba/api/documents/9c8c526e...
8,2376_Treasury Mint FOP_CBA updated_Redacted.pdf,2376,Department of the Treasury,FOP- Fraternal Order of Police,NaN,12/05/2021,https://www.opm.gov/cba/api/documents/0d3c0948...
9,2672_DOI NPS Independence NHP & AFGE 2058_0812...,2058,Department of Defense,Laborers' International Union,1310,9/22/2005,https://www.opm.gov/cba/api/documents/c4aed959...


In [9]:
source_rows = joined.copy()

source_df = pd.DataFrame(
    {
        "title": source_rows["filename"],
        "pdf_url": source_rows["pdf_url"],
        "document_id": source_rows["document_id"],
        "filename": source_rows["filename"],
        "bus_code": source_rows["normalized_bus_code"],
        "agency": source_rows.get(agency_col),
        "union": source_rows.get(union_col),
        "local": source_rows.get(local_col),
        "expiration_date": source_rows.get(expiration_col),
        "source": "OPM",
    }
).drop_duplicates(subset=["pdf_url"])

source_df.to_csv(SOURCE_CSV, index=False)
source_df.head()

,title,pdf_url,document_id,filename,bus_code,agency,union,local,expiration_date,source
0,ARS Animal Disease Center2015-06-29BUS 1529.pdf,https://www.opm.gov/cba/api/documents/4ba838b6...,4ba838b6-8141-496d-981a-62382e7850fb,ARS Animal Disease Center2015-06-29BUS 1529.pdf,1529,Department of Agriculture,AFGE- American Federation of Government Employees,2315,6/29/2025,OPM
2,1085_FMCS & NAGE R3-118_11102020-redacted.pdf,https://www.opm.gov/cba/api/documents/c4ee5660...,c4ee5660-1499-e911-915b-005056a577c8,1085_FMCS & NAGE R3-118_11102020-redacted.pdf,1085,Federal Mediation And Conciliation Service,NAGE- National Association of Government Emplo...,R3-118,2/11/2025,OPM
3,2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf,https://www.opm.gov/cba/api/documents/79fa136d...,79fa136d-7297-e911-915b-005056a577c8,2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf,1968,Department of Defense,AFGE- American Federation of Government Employees,1858,10/17/2020,OPM
5,1555 & 1564_DOE AFGE 788_06302019- redacted.pdf,https://www.opm.gov/cba/api/documents/5f1837cd...,5f1837cd-caa7-e911-915c-005056a577c8,1555 & 1564_DOE AFGE 788_06302019- redacted.pdf,1555,Department of Energy,AFGE- American Federation of Government Employees,788,9/28/2023,OPM
7,3708_Army_USACE_CBA_02022020.pdf,https://www.opm.gov/cba/api/documents/9c8c526e...,9c8c526e-f9fb-e911-9160-005056a577c8,3708_Army_USACE_CBA_02022020.pdf,3708,Department of Defense,AFGE- American Federation of Government Employees,2187,2/02/2020,OPM


In [10]:
HEADERS = {"User-Agent": "cba-clock-research-downloader/0.1"}


def sha256_bytes(content: bytes) -> str:
    return hashlib.sha256(content).hexdigest()


def safe_filename(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9._ -]", "", value)
    value = re.sub(r"\s+", "_", value)
    return value[:180]


def download_pdf(row: dict, overwrite: bool = False) -> dict:
    filename = safe_filename(row["filename"])
    output_file = RAW_PDF_DIR / filename

    result = {
        "pdf_url": row["pdf_url"],
        "filename": filename,
        "output_path": str(output_file.relative_to(PROJECT_ROOT)),
        "status": None,
        "bytes": None,
        "sha256": None,
        "content_type": None,
        "error": None,
    }

    if output_file.exists() and not overwrite:
        content = output_file.read_bytes()
        result["status"] = "skipped_existing"
        result["bytes"] = len(content)
        result["sha256"] = sha256_bytes(content)
        return result

    try:
        response = requests.get(row["pdf_url"], headers=HEADERS, timeout=60)
        response.raise_for_status()

        content = response.content
        result["content_type"] = response.headers.get("content-type")
        result["bytes"] = len(content)

        if not content.startswith(b"%PDF"):
            result["status"] = "failed_not_pdf"
            result["error"] = "Response did not start with %PDF"
            return result

        output_file.write_bytes(content)
        result["status"] = "downloaded"
        result["sha256"] = sha256_bytes(content)
        return result

    except Exception as e:
        result["status"] = "failed"
        result["error"] = repr(e)
        return result


results = []
for idx, row in enumerate(source_df.to_dict("records"), start=1):
    print(f"[{idx}/{len(source_df)}] {row['filename']}")
    result = download_pdf(row)
    print(result["status"], result.get("error") or "")
    results.append(result)
    time.sleep(0.5)

manifest_df = pd.DataFrame(results)
manifest_df.to_csv(MANIFEST_CSV, index=False)
manifest_df

[1/10] ARS Animal Disease Center2015-06-29BUS 1529.pdf
downloaded 
[2/10] 1085_FMCS & NAGE R3-118_11102020-redacted.pdf
downloaded 
[3/10] 2679_DOT SLSDC & AFGE 1968_09302021-redacted.pdf
downloaded 
[4/10] 1555 & 1564_DOE AFGE 788_06302019- redacted.pdf
downloaded 
[5/10] 3708_Army_USACE_CBA_02022020.pdf
downloaded 
[6/10] 2376_Treasury Mint FOP_CBA updated_Redacted.pdf
downloaded 
[7/10] 2672_DOI NPS Independence NHP & AFGE 2058_08122019.pdf
downloaded 
[8/10] 1012_DOI BIA & Indian Educators Federation 4524_10042019- redacted.pdf
downloaded 
[9/10] 1501_Treasury BEP & IPPDE Local 32 Engravers_05142022-redacted.pdf
downloaded 
[10/10] 1010 1120 HUD & AFGE- redacted.pdf
downloaded 


,pdf_url,filename,output_path,status,bytes,sha256,content_type,error
0,https://www.opm.gov/cba/api/documents/4ba838b6...,ARS_Animal_Disease_Center2015-06-29BUS_1529.pdf,data/samples/raw_pdfs/ARS_Animal_Disease_Cente...,downloaded,492244,b20178ad513e3d8b57bf816ceda397e0471000a49d3f1d...,application/octet-stream,None
1,https://www.opm.gov/cba/api/documents/c4ee5660...,1085_FMCS_NAGE_R3-118_11102020-redacted.pdf,data/samples/raw_pdfs/1085_FMCS_NAGE_R3-118_11...,downloaded,470216,653d712d6ee8801b792870d4b832296b4bd9b8ea46326e...,application/octet-stream,None
2,https://www.opm.gov/cba/api/documents/79fa136d...,2679_DOT_SLSDC_AFGE_1968_09302021-redacted.pdf,data/samples/raw_pdfs/2679_DOT_SLSDC_AFGE_1968...,downloaded,611577,7cafc998c900c86e1c682d3458153357d97d6bf2363f10...,application/octet-stream,None
3,https://www.opm.gov/cba/api/documents/5f1837cd...,1555_1564_DOE_AFGE_788_06302019-_redacted.pdf,data/samples/raw_pdfs/1555_1564_DOE_AFGE_788_0...,downloaded,5636029,68159e0ff9af7f13e95d5d6f79ed267220c8773e1fecc7...,application/octet-stream,None
4,https://www.opm.gov/cba/api/documents/9c8c526e...,3708_Army_USACE_CBA_02022020.pdf,data/samples/raw_pdfs/3708_Army_USACE_CBA_0202...,downloaded,379123,55b6a375696b282ef3d77a657197f33b22dd70ffb6e0f1...,application/octet-stream,None
5,https://www.opm.gov/cba/api/documents/0d3c0948...,2376_Treasury_Mint_FOP_CBA_updated_Redacted.pdf,data/samples/raw_pdfs/2376_Treasury_Mint_FOP_C...,downloaded,1021734,c6e616c0bb86fcba2488eea89736850027fd57c8ee49b8...,application/octet-stream,None
6,https://www.opm.gov/cba/api/documents/c4aed959...,2672_DOI_NPS_Independence_NHP_AFGE_2058_081220...,data/samples/raw_pdfs/2672_DOI_NPS_Independenc...,downloaded,12976490,01dde432036783ba94fcb72bde545291ac395502415b2a...,application/octet-stream,None
7,https://www.opm.gov/cba/api/documents/622e1554...,1012_DOI_BIA_Indian_Educators_Federation_4524_...,data/samples/raw_pdfs/1012_DOI_BIA_Indian_Educ...,downloaded,23663658,6b5dbdf5c0b1e4a5aa1d96da5f42a9aaf01263f6f00d8f...,application/octet-stream,None
8,https://www.opm.gov/cba/api/documents/7512a833...,1501_Treasury_BEP_IPPDE_Local_32_Engravers_051...,data/samples/raw_pdfs/1501_Treasury_BEP_IPPDE_...,downloaded,213670,1fd8cbe8d9f486bc20500852c50cac5b8d9bd20f935a3c...,application/octet-stream,None
9,https://www.opm.gov/cba/api/documents/6acdd03c...,1010_1120_HUD_AFGE-_redacted.pdf,data/samples/raw_pdfs/1010_1120_HUD_AFGE-_reda...,downloaded,1329621,6ca8b81a10b57cbd689593866c20f41d35b57d1ac307b1...,application/octet-stream,None
